 # Práctica de aprendizaje automático (parte 4)

# Preparar un modelo de aprendizaje automático para explotación (3 puntos)

Una vez finalizado tus estudios de grado, estás planeando emprender con tu propia startup. Estás considerando tres sectores en los cuales podrías desarrollar un modelo de aprendizaje automático que te permitiría establecer tu empresa en ese ámbito. Los sectores que tienes en mente son:

 - Predicción de las calificaciones ESRB para videojuegos.
 - Predicción de la experiencia de los usuarios de gimnasios.
 - Predicción de adopción de mascotas.

Para ello, se te proporcionan tres conjuntos de datos etiquetados (en la carpeta `data/`) que corresponden a cada sector:

 - pet_adoption_construccion.csv
 - gym_members_expertise_construccion.csv
 - videogame_esrb_ratings_construccion.csv

Tu tarea es entrenar un modelo de aprendizaje automático utilizando **uno** de los datasets proporcionados. Aplica tus conocimientos adquiridos en esta práctica para obtener el mejor modelo posible. 

Una vez que hayas finalizado el modelo, deberás utilizarlo para clasificar los datos del conjunto de explotación correspondiente al dataset elegido. Estos datasets de explotación son similares a los de construcción, pero **la columna de la clase solo tiene NaNs**.

Los conjuntos de datos de explotación (en la carpeta `data/`) son:

 - pet_adoption_explotacion.csv
 - gym_members_expertise_explotacion.csv
 - videogame_esrb_ratings_explotacion.csv

Utilizando el dataset de explotación debes construir un fichero de predicciones. Junto a este notebook debes entregar este fichero. Guarda el fichero de predicciones generado en la carpeta `predicciones/`. Los posibles nombres para este fichero son:

 - pet_adoption_predicciones.csv
 - gym_members_expertise_predicciones.csv
 - videogame_esrb_ratings_predicciones.csv

El nombre de tu fichero dependerá del dataset elegido. **Atención: si se entrega un fichero de predicciones incorrecto no se podrá evaluar**. El fichero a entregar debe tener dos columnas: el ID del ítem ("`id`") y la predicción para ese ítem ("`prediccion`"). Los ficheros de la carpeta `predicciones/`:

 - pet_adoption_ejemplo_fichero_predicciones.csv
 - gym_members_expertise_ejemplo_fichero_predicciones.csv
 - videogame_esrb_ratings_ejemplo_fichero_predicciones.csv

son ejemplos que ya tienen el formato solicitado para el archivo de entrega. Recuerda que el fichero de predicciones entregado debe acabar en **_predicciones.csv** y no en **_ejemplo_fichero_predicciones.csv**.

Recuerda explicar detalladamente los pasos que has seguido, las decisiones que has tomado y las conclusiones obtenidas. Se recomienda el uso de gráficos y tablas para respaldar tus argumentos y resultados (**recuerda incluir las barras de error en las gráficas y el error estándar en las tablas**).

Puedes encontrar información sobre cada dataset en los siguientes archivos (en la carpeta `data/`):

 - pet_adoption_info.txt
 - gym_members_expertise_info.txt
 - videogame_esrb_ratings_info.txt

**Importante**: Debes utilizar uno de los clasificadores con los que se ha trabajado en los apartados previos de la práctica. No está permitido utilizar otros clasificadores ni modelos de paquetes diferentes a **scikit-learn** (sklearn).

**Importante**: Solo se debe entregar un único archivo con todas las predicciones. Si se entregan varios archivos de predicciones para diferentes conjuntos de datos, el corrector automático seleccionará uno al azar. Esto significa que podríais perder toda la puntuación asociada a las predicciones si el archivo elegido no es el correcto Por ello, entregad un único archivo con predicciones.

En cumplimiento con el REGLAMENTO DE EVALUACIÓN ACADÉMICA DE LA ESCUELA POLITÉCNICA SUPERIOR DE LA UNIVERSIDAD DE AUTÓNOMA DE MADRID, artículo 14; "En el caso de copia, la asignatura se puntuará en la convocatoria donde se produjo la copia con cero puntos. Como medida adicional, el profesor puede iniciar un expediente informativo, de acuerdo con el Reglamento de Evaluación de la UAM".

In [3]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import cross_val_score

In [4]:
datos_construccion = pd.read_csv("./data/videogame_esrb_ratings_construccion.csv")
datos_explotacion  = pd.read_csv("./data/videogame_esrb_ratings_explotacion.csv")

In [5]:
datos_construccion.columns

Index(['id', 'alcohol_reference', 'blood_or_gore', 'cartoon_violence',
       'crude_humor', 'drug_reference', 'fantasy_violence', 'intense_violence',
       'language', 'lyrics', 'mild_blood', 'mild_cartoon_violence',
       'mild_fantasy_violence', 'mild_language', 'mild_lyrics',
       'mild_suggestive_themes', 'mild_violence', 'no_descriptors',
       'sexual_content', 'sexual_themes', 'simulated_gambling',
       'strong_janguage', 'strong_sexual_content', 'suggestive_themes',
       'violence', 'ESRB_Rating'],
      dtype='object')

In [6]:
datos_explotacion.columns

Index(['id', 'alcohol_reference', 'blood_or_gore', 'cartoon_violence',
       'crude_humor', 'drug_reference', 'fantasy_violence', 'intense_violence',
       'language', 'lyrics', 'mild_blood', 'mild_cartoon_violence',
       'mild_fantasy_violence', 'mild_language', 'mild_lyrics',
       'mild_suggestive_themes', 'mild_violence', 'no_descriptors',
       'sexual_content', 'sexual_themes', 'simulated_gambling',
       'strong_janguage', 'strong_sexual_content', 'suggestive_themes',
       'violence', 'ESRB_Rating'],
      dtype='object')

In [7]:
# la columna a predecir es "ESRB_Rating" y los valores están en el rango [0-4]
# por tanto, tenemos 5 clases en total

# Esta columna se sabe en el dataset de construcción:
datos_construccion["ESRB_Rating"].values[:10]

array([2, 2, 3, 3, 2, 2, 0, 0, 0, 2])

In [8]:
# Sin embargo no se sabe en el dataset de explotación (clientes actuales):
datos_explotacion["ESRB_Rating"].values[:10]

array([nan, nan, nan, nan, nan, nan, nan, nan, nan, nan])

Nuestro conjunto de entrenamiento va a ser el DataFrame datos_construcción, para poder hacer después la predicciones sobre datos_explotacion. Vamos a comprobar que no haya datos vacíos en ninguno de los atributos de los dos conjuntos de datos:

In [15]:
print('Conjunto de entrenamiento: \n', datos_construccion.info())
print('Conjunto de predicción: \n', datos_explotacion.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1451 entries, 0 to 1450
Data columns (total 26 columns):
 #   Column                  Non-Null Count  Dtype
---  ------                  --------------  -----
 0   alcohol_reference       1451 non-null   int64
 1   blood_or_gore           1451 non-null   int64
 2   cartoon_violence        1451 non-null   int64
 3   crude_humor             1451 non-null   int64
 4   drug_reference          1451 non-null   int64
 5   fantasy_violence        1451 non-null   int64
 6   intense_violence        1451 non-null   int64
 7   language                1451 non-null   int64
 8   lyrics                  1451 non-null   int64
 9   mild_blood              1451 non-null   int64
 10  mild_cartoon_violence   1451 non-null   int64
 11  mild_fantasy_violence   1451 non-null   int64
 12  mild_language           1451 non-null   int64
 13  mild_lyrics             1451 non-null   int64
 14  mild_suggestive_themes  1451 non-null   int64
 15  mild_violence        

In [10]:
datos_construccion.head(5)

,id,alcohol_reference,blood_or_gore,cartoon_violence,crude_humor,drug_reference,fantasy_violence,intense_violence,language,lyrics,...,mild_violence,no_descriptors,sexual_content,sexual_themes,simulated_gambling,strong_janguage,strong_sexual_content,suggestive_themes,violence,ESRB_Rating
0,100002,0,1,0,1,0,0,0,0,1,...,0,0,0,0,0,0,0,0,1,2
1,100003,0,0,0,0,1,0,0,0,0,...,0,1,0,0,0,0,0,1,1,2
2,100005,1,0,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,3
3,100006,0,0,0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,3
4,100007,0,1,0,0,0,0,0,1,0,...,0,0,0,1,0,0,0,0,0,2


No hay ningún atributo con datos NaN (excepto los ratings ESBR a predecir), por tanto, no hay que hacer sustitución de datos perdidos. El atributo que vamos a quitar porque sólo va a producir ruido es 'id', que es un identificador único de cad juego que no tiene nada que ver con el rating ESBR.

In [12]:
class_label = 'ESRB_Rating'
#como los ID son únicos a cada entrada, es un atributo despreciable
del datos_construccion['id']
feature_names = list(datos_construccion.columns)
feature_names.remove(class_label)
print(feature_names)
X = df[feature_names].values
y = df[class_label].values

['alcohol_reference', 'blood_or_gore', 'cartoon_violence', 'crude_humor', 'drug_reference', 'fantasy_violence', 'intense_violence', 'language', 'lyrics', 'mild_blood', 'mild_cartoon_violence', 'mild_fantasy_violence', 'mild_language', 'mild_lyrics', 'mild_suggestive_themes', 'mild_violence', 'no_descriptors', 'sexual_content', 'sexual_themes', 'simulated_gambling', 'strong_janguage', 'strong_sexual_content', 'suggestive_themes', 'violence']


Vamos a entrenar un MLP y después lo vamos a comparar con un árbol de decisión para ver con cuál haremos la predicción. Para ello primero vamos a proceder preprocesar los datos. Como los atributos son numéricos discretos, actuando como buleanos (pues solo son unos y ceros), aquí tal vez merezca la pena derivar un nuevo atributo que sea la suma de cada atributo y que actúe como un indicador del posible rating, aunque antes consideramos que sería ideal ver qué atributos son los que más pueden influir en el rating.

In [13]:
from sklearn.neural_network import MLPClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.model_selection import cross_val_score
import math

trainingSample = math.floor(1451/10 * 10 - 1451/10) #la división es entre 10 porque vamos evaluar a través de cross_val_score con 10 k-folds
prof_max = math.floor(math.log2(trainingSample))

n_features = range(1, len(feature_names)+1)
for n in n_features:
    select_k_best = SelectKBest(score_func=chi2, k=n)
    X_k_best_chi2 = select_k_best.fit_transform(X, y)

    
#creamos al atributo derivado:
datos_construccion['Score'] = datos_construccion[feature_names].sum(axis=1)
